### Beamsearch

[Video](https://youtu.be/w0Vr9kq99MU)

Falls A* zu langsam ist, können wir versuchen, mit Beamsearch eine einigermaßen gute Lösung zu finden. Wir benötigen
dazu wieder eine Heuristik. Sie muss aber nicht zulässig sein, d.h. sie darf die tatsächlichen Vorwärtskosten auch
überschätzen. Sie sollte nur einigermaßen in Richtung Ziel zeigen.

Wir geben uns eine Zahl *beamwidth* (Strahlbreite) vor. Beamsearch arbeitet wie die Breitensuche Ebene für Ebene.
Nachdem eine Ebene untersucht ist, werden von den gefundenen Folgezuständen aber nur die besten *beamwidth* Zustände
behalten und weiter untersucht. Die Suche wird also nicht immer breiter, sondern ist höchstens *beamwidth* breit. Man
kann sich das wie eine Taschenlampe vorstellen, die nur einen Teil des Unbekannten ausleuchtet. Zeigt die Heuristik in
Richtung Ziel, wandert der Lichtstrahl dorthin.

<img src='beamsearch.png'>

Der Preis: Der kürzeste Weg kann außerhalb des Strahls liegen. Beamsearch findet deshalb nicht immer den kürzesten Weg
und manchmal gar keinen.

#### Der Algorithmus

Die frontier ist eine normale Liste mit den Zuständen der aktuellen Ebene. Die Folgezustände sammeln wir zuerst in
einer zweiten Liste `nextfrontier`. Ist die Ebene fertig, sortieren wir `nextfrontier` nach der Heuristik und behalten
die ersten `beamwidth` Zustände als neue frontier.

In [1]:
def beamsearch(startstate, beamwidth):
    frontier = [startstate]
    prev = {startstate: None}
    nextfrontier = []
    while frontier:
        for state in frontier:
            if goaltest(state):
                return prev, state
            for v in nextstates(state):
                if v not in prev:
                    prev[v] = state
                    nextfrontier.append(v)
        frontier = sorted(nextfrontier, key=h)
        frontier = frontier[:beamwidth]
        nextfrontier = []
    return None, None

def reconstructPath(prev, goalstate):
    state = goalstate
    path = []
    while state is not None:
        path.append(state)
        state = prev[state]
    path.reverse()
    return path

`sorted(nextfrontier, key=h)` sortiert die Zustände nach ihrem Heuristik-Wert, der beste kommt zuerst. Zustände mit
gleichem Wert bleiben in der Reihenfolge, in der sie gefunden wurden.

Lässt man das Sortieren und Abschneiden weg (`frontier = nextfrontier`), erhält man wieder die Breitensuche, diesmal
ohne Warteschlange.

Alle gefundenen Zustände stehen in `prev`, auch die, die beim Abschneiden herausgefallen sind. Sie werden später nicht
noch einmal in die frontier aufgenommen.

#### Beispiel: Labyrinth

Wir nehmen die beiden Labyrinthe aus dem Notebook zu A*, mit der Manhattan-Distanz als Heuristik.

In [2]:
def lade(dateiname):
    global grid, height, width, start, ziel
    f = open(dateiname)
    grid = f.read().splitlines()
    f.close()
    height = len(grid)
    width = len(grid[0])
    for x in range(height):
        for y in range(width):
            if grid[x][y] == 'S':
                start = (x, y)
            elif grid[x][y] == 'Z':
                ziel = (x, y)

dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]    # oben, unten, links, rechts

def nextstates(state):
    x, y = state
    tmp = []
    for xd, yd in dirs:
        x1 = x + xd
        y1 = y + yd
        if 0 <= x1 < height and 0 <= y1 < width and grid[x1][y1] != '#':
            tmp.append((x1, y1))
    return tmp

def goaltest(state):
    return state == ziel

def h(state):
    x, y = state
    x0, y0 = ziel
    return abs(x0 - x) + abs(y0 - y)

def zeige(prev, goal):
    path = reconstructPath(prev, goal)
    bild = [list(zeile) for zeile in grid]
    for x, y in prev:
        if bild[x][y] == ' ':
            bild[x][y] = '.'
    for x, y in path[1:-1]:
        bild[x][y] = 'o'
    for zeile in bild:
        print(''.join(zeile))
    print('Weglänge:', len(path) - 1, ' gefundene Zustände:', len(prev))

In [3]:
lade('maze3.txt')
prev, goal = beamsearch(start, 2)
zeige(prev, goal)

##########
#  ..... #
#  .ooooZ#
#  .S.####
#   ..   #
#        #
##########
Weglänge: 5  gefundene Zustände: 16


In der Falle aus dem Notebook zu A* hängt das Ergebnis von der Strahlbreite ab:

In [4]:
lade('maze4.txt')
for beamwidth in [1, 2]:
    prev, goal = beamsearch(start, beamwidth)
    print('beamwidth =', beamwidth)
    zeige(prev, goal)
    print()

beamwidth = 1
############
#          #
# ########.#
# #oooooooo#
# #o######o#
# #oooooo#o#
#.######o#o#
#Sooooooo#Z#
############
Weglänge: 27  gefundene Zustände: 30

beamwidth = 2
############
#oooooooooo#
#o########o#
#o#.......o#
#o#.######o#
#o#......#o#
#o######.#o#
#S.......#Z#
############
Weglänge: 21  gefundene Zustände: 44



Mit `beamwidth = 1` wird in jeder Ebene nur der eine beste Zustand behalten. Dann läuft Beamsearch wie Greedy zuerst
nach rechts, nimmt den langen Umweg und findet einen Weg mit 27 Zügen. Schon mit `beamwidth = 2` bleibt der Weg nach oben im Strahl, und
Beamsearch findet den kürzesten Weg mit 21 Zügen.

#### Beispiel: 8-Puzzle

Beim 8-Puzzle nehmen wir die Manhattan-Summe als Heuristik. A* hat damit die kürzeste Lösung mit 26 Zügen gefunden
und dabei 3081 Zustände gefunden.

In [5]:
def puzzle(n):
    '''definiert goaltest, nextstates und h für das Schiebepuzzle mit n mal n Feldern'''
    global goaltest, nextstates, h
    ziel = tuple(range(n * n))

    def goaltest(state):
        return state == ziel

    def nextstates(state):
        tmp = []
        i = state.index(0)
        for d in [-n, n, -1, 1]:
            j = i + d
            if 0 <= j < n * n and (i % n == j % n or i // n == j // n):
                a = list(state)
                a[i], a[j] = a[j], a[i]
                tmp.append(tuple(a))
        return tmp

    def h(state):
        summe = 0
        for i in range(n * n):
            zahl = state[i]
            if zahl != 0:
                summe += abs(i // n - zahl // n) + abs(i % n - zahl % n)
        return summe

puzzle(3)
startstate = (7, 2, 4, 5, 0, 6, 8, 3, 1)
for beamwidth in [1, 10, 100, 1000]:
    prev, goal = beamsearch(startstate, beamwidth)
    if goal is None:
        print(f'beamwidth {beamwidth:>5}: keine Lösung')
    else:
        print(f'beamwidth {beamwidth:>5}: {len(reconstructPath(prev, goal)) - 1} Züge, {len(prev)} gefundene Zustände')

beamwidth     1: keine Lösung
beamwidth    10: 32 Züge, 512 gefundene Zustände
beamwidth   100: 26 Züge, 3287 gefundene Zustände
beamwidth  1000: 26 Züge, 23321 gefundene Zustände


Mit `beamwidth = 1` gibt es keine Lösung: Irgendwann sind alle Folgezustände des einen behaltenen Zustands schon in
`prev`, die frontier wird leer. Mit 10 findet Beamsearch eine Lösung mit 32 Zügen, ab 100 die kürzeste mit 26 Zügen.
Für das 8-Puzzle lohnt sich Beamsearch nicht, A* ist hier schnell genug.

#### Beispiel: 15-Puzzle

Beim 15-Puzzle mit $4 \times 4$ Feldern gibt es etwa $10^{13}$ erreichbare Zustände. Wir mischen das Puzzle mit 200
zufälligen Zügen.

In [6]:
import random

puzzle(4)
random.seed(1)
state = tuple(range(16))
for i in range(200):
    state = random.choice(nextstates(state))
startstate = state
print(startstate, ' Manhattan-Summe:', h(startstate))

(8, 3, 15, 2, 1, 13, 4, 12, 14, 9, 10, 6, 5, 0, 7, 11)  Manhattan-Summe: 32


Zuerst A* aus dem Notebook zu A*. Wir brechen ab, sobald 200000 Zustände gefunden sind:

In [7]:
from heapq import heappop, heappush
import time

def astar(startstate, grenze):
    frontier = [(h(startstate), startstate)]
    prev = {startstate: None}
    g = {startstate: 0}
    while frontier and len(prev) < grenze:
        _, state = heappop(frontier)
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            gg = g[state] + 1
            if v not in prev or gg < g[v]:
                g[v] = gg
                heappush(frontier, (g[v] + h(v), v))
                prev[v] = state
    return prev, None

t = time.time()
prev, goal = astar(startstate, 200000)
print('Lösung gefunden:', goal is not None, ' Zeit:', round(time.time() - t, 1), 's')

Lösung gefunden: False  Zeit: 1.2 s


A* hat nach 200000 Zuständen noch keine Lösung. Selbst nach 2 Millionen Zuständen und etwa 16 Sekunden ist es noch
nicht fertig. Beamsearch findet in kurzer Zeit eine Lösung:

In [8]:
for beamwidth in [10, 100, 1000, 10000]:
    t = time.time()
    prev, goal = beamsearch(startstate, beamwidth)
    zuege = len(reconstructPath(prev, goal)) - 1
    print(f'beamwidth {beamwidth:>5}: {zuege} Züge, {len(prev)} gefundene Zustände, {time.time() - t:.1f} s')

beamwidth    10: 148 Züge, 3090 gefundene Zustände, 0.0 s
beamwidth   100: 80 Züge, 15490 gefundene Zustände, 0.1 s


beamwidth  1000: 60 Züge, 105072 gefundene Zustände, 0.4 s


beamwidth 10000: 52 Züge, 804384 gefundene Zustände, 3.6 s


Je größer die Strahlbreite, desto kürzer der gefundene Weg, aber desto länger dauert die Suche. Ob 52 Züge die
kürzeste Lösung sind, wissen wir nicht. Beamsearch liefert eine gute Lösung, aber keine Garantie.

#### Die Heuristik bei Beamsearch

Bei Beamsearch kommt es nur auf die **Reihenfolge** an, in die die Heuristik die Zustände bringt. Ob man die
Manhattan-Summe nimmt oder ihr Doppeltes, ergibt dieselbe Sortierung und damit dasselbe Ergebnis. Deshalb muss die
Heuristik nicht zulässig sein. Sie sollte nur gute Zustände vor schlechte sortieren.

| Verfahren | frontier | Heuristik | findet |
|---|---|---|---|
| Breitensuche | Warteschlange, Ebene für Ebene | keine | sicher einen kürzesten Weg |
| A* | Heap nach $g + h$ | zulässig | sicher einen kürzesten Weg |
| Greedy | Heap nach $h$ | beliebig | irgendeinen Weg, oft schnell |
| Beamsearch | die besten *beamwidth* Zustände jeder Ebene | beliebig | meist einen guten Weg, manchmal keinen |

#### Das Beamsearch-Muster

Wie bei der Breitensuche und bei A* bleiben `beamsearch`, `reconstructPath` und `getMoves` gleich. Zu schreiben sind
die Modellierung des Zustands, `nextstates`, `goaltest`, die Heuristik `h` und falls gewünscht `getMove`.

```python
def h(state):
    '''returns: die geschätzten Vorwärtskosten, dürfen auch überschätzen'''

# Aufruf:
BEAMWIDTH = 1000
prev, goal = beamsearch(startstate, BEAMWIDTH)
path = reconstructPath(prev, goal)
print(getMoves(path))
```